In [ ]:
from labdata.schema import (
    Dataset,
    DatasetEvents,
    SpikeSorting,
    UnitMetrics,
    EphysRecording,
    UnitCount,
)

# from labdata import chipmunk
# from chipmunk import Chipmunk

from spks.event_aligned import population_peth
from spks.viz import plot_event_aligned_raster

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display, clear_output
from alive_progress import alive_bar

%matplotlib widget

In [ ]:
subject = "GRB058"
session = "20260224_152424"

obx_sessions = (DatasetEvents() & EphysRecording() & 'stream_name = "obx"').proj()
sess_query = (
    obx_sessions & f'subject_name = "{subject}"' & f'session_name = "{session}"'
)
sess_query

In [ ]:
good_unit_ids = tuple(
    (sess_query * UnitCount.Unit & "unit_criteria_id = 1" & "passes = 1").fetch(
        "subject_name", "session_name", "unit_id", as_dict=True
    )
)
good_units = pd.DataFrame(
    ((SpikeSorting.Unit & good_unit_ids) * UnitMetrics).fetch(
        "unit_id", "spike_times", "depth", as_dict=True
    )
)

In [ ]:
dset = (Dataset() & sess_query).proj()
events = DatasetEvents.Digital() & dset
events = pd.DataFrame(events.fetch_synced())

In [ ]:
align_ev = {
    "stim": events.query("event_name == '0'").event_timestamps.values[0],
    "trial_start": events.query("event_name == '2'").event_timestamps.values[0],
    "frames": events.query("event_name == '3'").event_timestamps.values[0],
    "left_port": events.query("event_name == '4'").event_timestamps.values[0],
    "center_port": events.query("event_name == '5'").event_timestamps.values[0],
    "right_port": events.query(
        "event_name == '6' & stream_name == 'obx'"
    ).event_timestamps.values[0],
}

stim = align_ev["stim"]
# Each pulse is 15ms with ≥25ms gap → real onset-to-onset ≥ 40ms
# Take the first raw onset after each >25ms gap (= first onset of each pulse)
stim_ev = np.concatenate([[stim[0]], stim[1:][np.diff(stim) > 0.025]])
# First stim per trial: first onset after a >500ms gap
first_stim_ev = np.concatenate([[stim[0]], stim[1:][np.diff(stim) > 1]])
align_ev.update({"stim_ev": stim_ev, "first_stim_ev": first_stim_ev})

In [ ]:
plt.figure(figsize=(15, 5))
plt.vlines(align_ev["stim"], ymin=-2, ymax=-1, color="royalblue", label="raw_stim")
plt.vlines(align_ev["stim_ev"], ymin=-1, ymax=0, color="dodgerblue", label="stim")
plt.vlines(align_ev["first_stim_ev"], ymin=0, ymax=1, color="blue", label="first_stim")
plt.vlines(
    align_ev["trial_start"], ymin=1, ymax=2, color="fuchsia", label="trial start"
)
plt.vlines(
    align_ev["left_port"], ymin=2, ymax=3, color="mediumseagreen", label="left port"
)
plt.vlines(align_ev["center_port"], ymin=3, ymax=4, color="red", label="center port")
plt.vlines(align_ev["right_port"], ymin=4, ymax=5, color="gold", label="right port")
plt.legend()
plt.xlim((200, 300))

In [ ]:
pre_seconds = 0.05
post_seconds = 0.1
binwidth_ms = 10
srate = float((EphysRecording.ProbeSetting() & sess_query).fetch("sampling_rate")[0])

In [ ]:
st_per_unit_s = [st / srate for st in good_units["spike_times"]]

with plt.ioff():
    fig, ax = plt.subplots(1, figsize=(5, 5))

unit_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(good_units) - 1,
    step=1,
    description="Unit:",
    continuous_update=False,
)

event_dropdown = widgets.Dropdown(
    options=list(align_ev.keys()),
    value="first_stim_ev",
    description="Event:",
)

bar_output = widgets.Output()


def update_plot(change):
    selected_unit = unit_slider.value
    selected_event = event_dropdown.value

    ax.cla()
    plot_event_aligned_raster(
        event_times=align_ev[selected_event],
        spike_times=st_per_unit_s[selected_unit],
        pre_seconds=pre_seconds,  # type: ignore
        post_seconds=post_seconds,  # type: ignore
        ax=ax,
    )

    ax.set_xlim(-pre_seconds, post_seconds)

    ymin_plot, ymax_plot = ax.get_ylim()
    ax.vlines(x=0, ymin=ymin_plot, ymax=ymax_plot, colors="r", linestyles="--")
    ax.set_ylabel("events")
    ax.set_xlabel("time from event (s)")
    ax.set_title(f"Unit {selected_unit} aligned to {selected_event}")
    fig.tight_layout()
    fig.canvas.draw_idle()


unit_slider.observe(update_plot, names="value")
event_dropdown.observe(update_plot, names="value")

display(widgets.VBox([unit_slider, event_dropdown, fig.canvas]))
update_plot(None)

In [ ]:
event_selector = widgets.Dropdown(
    options=align_ev.keys(),
    value="first_stim_ev",
    description="Event:",
)

bar_output = widgets.Output()

with plt.ioff():
    fig_psth, ax_psth = plt.subplots(1, figsize=(3, 6))


def update_plot(change):
    selected_event = event_selector.value

    with bar_output:
        clear_output(wait=True)
        with alive_bar(1, title="Computing PSTH", force_tty=True, length=30) as bar:
            psth, _, _ = population_peth(
                all_spike_times=good_units["spike_times"] / srate,
                alignment_times=align_ev[selected_event],
                pre_seconds=pre_seconds,
                post_seconds=post_seconds,
                binwidth_ms=binwidth_ms,
                kernel=None,
                pad=0,
            )
            bar()

    fig_psth.clf()
    ax = fig_psth.add_subplot(111)

    peak_indices = np.argmax(psth.mean(axis=1), axis=1)
    sorted_indices = np.argsort(peak_indices)
    kernel = np.mean(psth, axis=1)[sorted_indices, :]

    im = ax.imshow(kernel, aspect="auto", clim=(0, 30), cmap="grey_r")
    cbar = fig_psth.colorbar(im, ax=ax)
    cbar.set_label("sp/s")

    n_ticks = 5
    tick_step = (post_seconds + pre_seconds) / n_ticks
    binwidth_s = binwidth_ms / 1000
    zero_bin = pre_seconds / binwidth_s

    tick_times = np.concatenate(
        [
            np.arange(0, -pre_seconds - 1e-9, -tick_step)[::-1],
            np.arange(tick_step, post_seconds + 1e-9, tick_step),
        ]
    )
    tick_times = tick_times[(tick_times >= -pre_seconds) & (tick_times <= post_seconds)]
    ticks = zero_bin + tick_times / binwidth_s
    labels = np.round(tick_times, 2)
    ax.set_xticks(ticks, labels, rotation=45)

    ax.set_xlabel("time from event (s)")
    ax.set_ylabel("units")
    ax.set_title(f"Aligned to: {selected_event}")

    ymin, ymax = ax.get_ylim()
    ax.vlines(zero_bin, ymin, ymax, "r", "--", alpha=0.0)

    fig_psth.tight_layout()
    fig_psth.canvas.draw_idle()


event_selector.observe(update_plot, names="value")
display(widgets.VBox([event_selector, bar_output, fig_psth.canvas]))
update_plot(None)